# Consultas no banco `cotas.db`

Notebook simples para rodar `SELECT`s no banco SQLite do projeto.
Se o banco ainda nao existir, rode `python db/criar_banco.py` antes.

In [6]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import config

conn = sqlite3.connect(config.DB_PATH)
print(f"Conectado em: {config.DB_PATH}")

Conectado em: C:\Users\CLIENTE\Documents\Sistema de Cotas - Simples\db\cotas.db


## Tabelas existentes e quantidade de linhas

In [7]:
tabelas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name",
    conn,
)["name"].tolist()

for tabela in tabelas:
    total = pd.read_sql_query(f"SELECT COUNT(*) AS total FROM {tabela}", conn)["total"][0]
    print(f"{tabela:<28} {total} linha(s)")

cda_edges                    81282 linha(s)
cda_nodes                    20518 linha(s)
competencias                 6 linha(s)
demonstracoes                29745 linha(s)
demonstracoes_ai             0 linha(s)
fatos_relevantes             0 linha(s)
fundos                       31515 linha(s)
logs_pipeline                8 linha(s)
monitoramento                0 linha(s)
prestadores_historico        136739 linha(s)


## Amostra de fundos

In [ ]:
pd.read_sql_query(
    """
    SELECT cnpj, denominacao_social, classe, situacao, classificacao_risco
    FROM fundos
    LIMIT 20
    """,
    conn,
)

## Distribuicao de classificacao de risco

In [ ]:
pd.read_sql_query(
    "SELECT classificacao_risco, COUNT(*) AS total FROM fundos GROUP BY classificacao_risco",
    conn,
)

## Historico de prestadores de um fundo especifico

Troque o CNPJ abaixo pelo que quiser consultar.

In [ ]:
cnpj_consulta = "00068305000135"

pd.read_sql_query(
    """
    SELECT tipo_prestador, nome, data_inicio, data_fim
    FROM prestadores_historico
    WHERE fundo_cnpj = ?
    ORDER BY tipo_prestador, data_inicio DESC
    """,
    conn,
    params=(cnpj_consulta,),
)

## Ultimas execucoes de pipeline

In [ ]:
pd.read_sql_query(
    """
    SELECT nome_pipeline, status, data_execucao, linhas_processadas,
           linhas_inseridas, linhas_atualizadas, competencia
    FROM logs_pipeline
    ORDER BY id DESC
    LIMIT 10
    """,
    conn,
)

## Consulta livre

Escreva qualquer `SELECT` aqui embaixo.

In [ ]:
sql = "SELECT * FROM fundos WHERE classificacao_risco = 'ALTO' LIMIT 10"
pd.read_sql_query(sql, conn)

In [ ]:
conn.close()